# Final Integration – Meme Caption Overlay on Cartoonized Images

This notebook integrates the full pipeline:

- Uses **U-GAT-IT 100K cartoonized images** (from `1b_image_processing.ipynb`)
- Generates **mood-based captions** using a fine-tuned GPT-2 model (from `2b_caption_generation.ipynb`)
- Renders text over images using PIL (from `3a_meme_text_rendering.ipynb`)

The goal is to produce modular, mood-aware cartoon memes by combining vision and language models.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from PIL import Image
from IPython.display import display
import torch
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# Project structure
PROJECT_ROOT = "/content/drive/MyDrive/cartoonify"

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "assets/memes")
MODEL_PATH = os.path.join(PROJECT_ROOT, "models/gpt2-mood-caption-v2")
IMAGE_SIZE = (256, 256)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Directories ready.")

## 1. Load Caption Generator
We reuse the fine-tuned GPT-2 model from earlier (`2b_caption_generation.ipynb`).

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, pipeline
import sys

# Append helper script path
sys.path.append(os.path.join(PROJECT_ROOT, "helpers"))
import caption_helpers as helpers

# Load fine-tuned model
def load_generator(model_path):
    model = GPT2LMHeadModel.from_pretrained(model_path)
    tokenizer = GPT2Tokenizer.from_pretrained(model_path)
    tokenizer.pad_token = tokenizer.eos_token  # Fix for GPT2
    return pipeline("text-generation", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

caption_generator = load_generator(MODEL_PATH)
print("Caption generator loaded.")

### 2. Meme Rendering with Captions
We render generated captions directly onto the cartoon images.

In [ ]:
import sys
sys.path.append(os.path.join(PROJECT_ROOT, "helpers"))

from meme_helpers import create_meme_direct_pil

In [ ]:
# For testing

"""
create_meme_direct_pil(
    image_path=os.path.join(CARTOON_DIR, "face01.jpg"),
    top_text="FEELING MOODY",
    bottom_text="BUT STILL CARTOON COOL",
    output_path=os.path.join(OUTPUT_DIR, "meme01.jpg"),
    size=(256, 256)
)
"""

In [ ]:
def generate_caption(mood: str = "neutral", max_length: int = 30) -> str:
    prompt = f"mood: {mood}\ncaption:"
    result = caption_generator(prompt, max_length=max_length, num_return_sequences=1, do_sample=True)[0]["generated_text"]
    return result.split("caption:")[-1].strip().split("\n")[0]

In [ ]:
mood = "confused"
generated_caption = generate_caption(mood)
print(f"Prompted mood: {mood}")
print(f"Generated caption: {generated_caption}")

In [1]:
# Load cartoonization model (UGAT-IT 100K)
sys.path.append(os.path.join(PROJECT_ROOT, "UGATIT"))

from UGATIT import UGATIT
from argparse import Namespace
from types import MethodType

def get_config():
    return Namespace(
        dataset='selfie2anime',
        phase='test',
        light=False,
        ch=64,
        n_res=6,
        n_dis=6,
        img_size=256,
        img_ch=3,
        batch_size=1,
        iteration=100000,
        result_dir='results',
        checkpoint_dir=os.path.join(PROJECT_ROOT, "models/vision/checkpoints"),
        log_dir='log',
        sample_dir='sample',
        device='cuda' if torch.cuda.is_available() else 'cpu',
        benchmark_flag=True,
        resume=None
    )

def infer_single_image(self, input_path, output_path):
    from torchvision import transforms
    img = PILImage.open(input_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((self.img_size, self.img_size)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    img_tensor = transform(img).unsqueeze(0).to(self.device)
    with torch.no_grad():
        output = self.genA2B(img_tensor)
        if isinstance(output, tuple):
            output = output[0]
        out = (output.squeeze(0).cpu().numpy().transpose(1, 2, 0) + 1) / 2
        out = (out * 255).clip(0, 255).astype("uint8")
        PILImage.fromarray(out).save(output_path)

def load_inference_model():
    config = get_config()
    model = UGATIT(config)
    model.build_model()
    ckpt = torch.load(os.path.join(config.checkpoint_dir, "genA2B_final.pth"), map_location=config.device)
    model.genA2B.load_state_dict(ckpt)
    model.genA2B.eval()
    model.infer_single_image = MethodType(infer_single_image, model)
    return model

inference_model = load_inference_model()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


NameError: name 'sys' is not defined

## 3. Meme Creation from Raw Human Face Images

We now apply the full pipeline to real human selfies by:
- Converting each selfie to cartoon using U-GAT-IT (100K)
- Generating a mood-aware caption using GPT-2
- Rendering the meme using our custom overlay function


In [ ]:
from glob import glob
import random

RAW_INPUT_DIR = os.path.join(PROJECT_ROOT, "assets/raw_humanface")
image_paths = sorted(glob(os.path.join(RAW_INPUT_DIR, "*.jpg")))
moods = ["confused", "joyful", "angry", "proud", "sarcastic", "gloomy"]

for i, img_path in enumerate(image_paths[:10]):
    # Step 1: Cartoonize
    cartoon_path = os.path.join(OUTPUT_DIR, f"cartoon_face_{i+1:02}.jpg")
    inference_model.infer_single_image(img_path, cartoon_path)

    # Step 2: Caption
    mood = random.choice(moods)
    caption = generate_caption(mood)

    # Step 3: Render Meme
    meme_path = os.path.join(OUTPUT_DIR, f"meme_{i+1:02}.jpg")
    create_meme_direct_pil(
        image_path=cartoon_path,
        top_text=mood.upper(),
        bottom_text=caption,
        output_path=meme_path,
        size=IMAGE_SIZE
    )

    display(PILImage.open(meme_path))

## Conclusion

This notebook completes our end-to-end creative generation pipeline by integrating:

- A **cartoonized image generator** (U-GAT-IT at 100K iterations)
- A **mood-conditioned caption generator** (fine-tuned GPT-2)
- A **custom meme renderer** using PIL

By combining independently trained vision and language models, we were able to generate expressive, emotionally aligned meme-style outputs.

This modular pipeline reflects a flexible and interpretable approach to multimodal generation.

**Future improvements could include:**
- Caption reranking or filtering
- Adding emoji/styling based on mood
- Using CLIP or LPIPS to score image-text alignment